In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42 # for pdfs
matplotlib.rcParams['svg.fonttype'] = 'none' # for svgs
import matplotlib.pyplot as plt
import seaborn as sns

import flexiznam as flz
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import rsof_plots
from cottage_analysis.plotting import style
from v1_depth_map.figure_utils.treadmill import (
    load_treadmill_population_neurons_df,
    compute_treadmill_rsof_bins,
    add_trial_average_rsof_columns,
)
from v1_depth_map.figure_utils.rsof_integration import (
    plot_angle_eccentricity_polar,
    plot_g2d_fit_schematic,
    SEMIMAJOR_COLOR,
    SEMIMINOR_COLOR,
)


In [ ]:
# Set default font to Arial
import matplotlib.font_manager as fm

# optional, can be None or the path to arial.ttf:
arial_font_path = (
    "/Volumes/BlackPasspo/v1_depth_map/processed/v1_manuscript_figures/fonts/arial.ttf"
)
# "/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf"
# set matplotlib options
if arial_font_path is not None:
    arial_prop = fm.FontProperties(fname=arial_font_path)
    plt.rcParams["font.family"] = arial_prop.get_name()
    plt.rcParams.update({"mathtext.default": "regular"})  # make math mode also Arial
    fm.fontManager.addfont(arial_font_path)

In [ ]:
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
from v1_depth_map.paths import get_figures_roots

READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

regenerate_cached_files = False  # set to True to regenerate_cached_files everything

In [ ]:
# Load the treadmill population data (all sessions with the treadmill protocol), used
# to look up example cells by roi_uid.
(
    neurons_df_treadmill,
    simul_df_treadmill_population,
    simul_df_spheres_population,
    valid_treadmill_sessions,
    treadmill_sessions,
) = load_treadmill_population_neurons_df(flexilims_session)

In [ ]:
# Trial-averaged treadmill g2d fits (one sample per trial rather than per imaging frame):
# derive the tuning-ellipse geometry and the empirical-null significance flags used by the
# population panel below. `_plateau` matches the `method="plateau"` onset detection used
# when loading the example session's trials.
TA = "_treadmill_trial_average_plateau"
MIN_SIGMA = add_trial_average_rsof_columns(
    neurons_df_treadmill, ta_suffix=TA, null_method="empirical"
)

# Population for the polar panel: significant 2D-Gaussian fit on the treadmill, among
# neurons whose depth tuning was established in CLOSED LOOP (`is_depth_neuron`, no
# suffix). The two criteria deliberately come from different protocols: depth tuning
# from closed loop, where the animal's own movement varies depth, and the RS/OF
# geometry from the treadmill. No eccentricity cut - eccentricity is the radial axis, so "ridge" fits
# (unbounded along one axis) simply land at eccentricity 1 rather than being hidden.
ndf_polar = neurons_df_treadmill[
    neurons_df_treadmill[f"rsof_test_rsq_closedloop_g2d{TA}_sig"]
    & neurons_df_treadmill["is_depth_neuron"].fillna(False)
].dropna(subset=[f"g2d_theta{TA}", f"g2d_eccentricity{TA}"])

# A near-circular tuning ellipse has no meaningful orientation, so only clearly
# elongated fits enter the orientation histogram and the von Mises mixture. They stay
# on the polar scatter (drawn in black there) rather than being dropped silently.
ECC_CUTOFF = 0.6
ecc_ok = ndf_polar[f"g2d_eccentricity{TA}"].astype(float) > ECC_CUTOFF

print(
    f"\n{len(ndf_polar)} neurons in the polar panel "
    f"({len(ndf_polar) / len(neurons_df_treadmill):.1%} of {len(neurons_df_treadmill)}), "
    f"{ndf_polar.session.nunique()} sessions"
)
print(
    f"{int(ecc_ok.sum())} with eccentricity > {ECC_CUTOFF} (orientation histogram/fit), "
    f"{int((~ecc_ok).sum())} below the cut"
)
print(ndf_polar.groupby("session").size().rename("n_neurons"))

In [ ]:
# Load the example session's treadmill (motorised wheel) trials, used for the two
# example-cell RS/OF tuning panels below.
EXAMPLE_SESSION = "PZAG17.3a_S20250402"
example_mouse, example_session = EXAMPLE_SESSION.split("_")

ndf, trials_df_tm, trials_df_sphere = pipeline_utils.load_treadmill_and_sphere_datasets(
    project,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
    tread_kwargs=dict(method="plateau"),
)

max_abs_rs2motor_diff_ratio = 0.3
rs_bins, of_bins, tick_dict = compute_treadmill_rsof_bins(trials_df_tm)

In [ ]:
# How many components does the orientation distribution need? Fit the axial von Mises
# mixture for k = 1 .. MAX_K and compare BIC. Orientation is circular with 180 deg
# periodicity, so the mixture is fit on the doubled angle (see
# v1_depth_map.figure_utils.von_mises). Same data (elongated fits only, `ecc_ok`), k
# range and seed as the figure cell below, so the k picked here is the one drawn there.
from v1_depth_map.figure_utils import von_mises as vm

MAX_K_SELECT = 5
ks = list(range(1, MAX_K_SELECT + 1))
angles_select = (
    ndf_polar.loc[ecc_ok, f"g2d_theta{TA}"].astype(float).dropna().to_numpy()
)


def wrap_axial_deg(deg):
    """Wrap an axial angle to [-45, 135), matching fit_gb.get_gaussian_angle."""
    return (np.asarray(deg) + 45) % 180 - 45


vm_sel = vm.model_selection(
    np.radians(angles_select), max_k=MAX_K_SELECT, seed=42, verbose=False
)
vm_bic = {k: vm_sel[k]["bic"] for k in ks}
k_vm_best = min(vm_bic, key=vm_bic.get)

fig_bic, ax_bic = plt.subplots(1, 1, figsize=(3.8, 2.6))
ax_bic.plot(ks, [vm_bic[k] for k in ks], "o-", color="#CC79A7")
ax_bic.axvline(k_vm_best, color="k", ls=":", lw=1)
ax_bic.set_xticks(ks)
ax_bic.set_xlabel("n components")
ax_bic.set_ylabel("BIC")
ax_bic.set_title(f"von Mises mixture: best k = {k_vm_best}", fontsize=9)
ax_bic.spines[["top", "right"]].set_visible(False)
fig_bic.tight_layout()

print(
    f"n = {len(angles_select)} neurons (eccentricity > {ECC_CUTOFF}), "
    f"k = {ks[0]} .. {ks[-1]}\n"
)

print("Axial von Mises mixture (fit on the doubled angle)")
for k in ks:
    print(f"  k={k}  BIC={vm_bic[k]:10.1f}" + ("   <-- best" if k == k_vm_best else ""))
r_best = vm_sel[k_vm_best]
# mu comes back on the doubled-angle circle, so it can land outside [-45, 135).
mu_deg = wrap_axial_deg(np.degrees(r_best["mu_k"]))
print(f"  components of best k={k_vm_best}:")
for i in np.argsort(mu_deg):
    print(
        f"    mu={mu_deg[i]:7.1f} deg"
        f"   kappa={r_best['kappa_k'][i]:6.2f}"
        f"   fraction={r_best['pi_k'][i]:.3f}"
    )

In [ ]:
# Suggest example cells for EXAMPLE_SESSION: significant treadmill RS/OF (g2d) fit and
# a gaussian angle close to 45 deg (i.e. tuned to the RS/OF ratio, so depth-tuned on the
# treadmill). Sorted by how close the angle is to 45 deg, then by fit quality.
target_angle = 90  # degrees
angle_tol = 15  # keep cells within +/- this of target_angle

candidates = neurons_df_treadmill[
    (neurons_df_treadmill.session == EXAMPLE_SESSION)
    & neurons_df_treadmill.rsof_neuron_treadmill
    & ((neurons_df_treadmill.g2d_theta_treadmill - target_angle).abs() < angle_tol)
].copy()
candidates["angle_dist"] = (candidates.g2d_theta_treadmill - target_angle).abs()
candidates = candidates.sort_values(
    ["angle_dist", "rsof_test_rsq_closedloop_g2d_treadmill"],
    ascending=[True, False],
)

print(f"{len(candidates)} candidate cells in {EXAMPLE_SESSION}")
display(
    candidates[
        [
            "roi_uid",
            "g2d_theta_treadmill",
            "g2d_eccentricity_treadmill",
            "rsof_test_rsq_closedloop_g2d_treadmill",
            "g2d_preferred_RS_treadmill",
            "g2d_preferred_OF_treadmill",
            "is_depth_neuron",
            "is_depth_neuron_treadmill",
            "preferred_depth_closedloop_crossval_treadmill",
        ]
    ].head(20)
)

In [ ]:
# Example cells with
# Nice kinda triangular cell "PZAG17.3a_S20250402_169"
# Diagonal cell "PZAG17.3a_S20250402_31"
# OF: PZAG17.3a_S20250402_309
# RS: PZAG17.3a_S20250402_134, "PZAG17.3a_S20250402_330"
# RS: "PZAG17.3a_S20250402_235"
EXAMPLE_CELL = "PZAG17.3a_S20250402_235"
EXAMPLE_CELL2 = "PZAG17.3a_S20250402_309"
EXAMPLE_CELL3 = "PZAG17.3a_S20250402_31"

# Cell whose 2D-Gaussian fit illustrates the fit parameters in the bottom row: strongly
# fit (R2 0.71), elongated (ecc 0.75) and oriented at 43 deg, with a preferred RS/OF
# well inside the sampled grid so the whole ellipse is visible.
EXAMPLE_CELL4 = "PZAG17.3a_S20250402_142"

In [ ]:
# Two rows.
# Top: 3 example cells (A, B, C), RS/OF tuning matrix then the stacked OF-tuning-by-RS panels
# (binned mean +/- bootstrap 95% CI and 1d gaussian fit of the trials).
# Bottom: what the 2D-Gaussian fit measures (definitions + an example fit) and the
# population distribution of tuning-ellipse orientation and eccentricity (D, E).
cm = 1 / 2.54
FIG_W_CM, FIG_H_CM = 18.5, 10.5
ASPECT = FIG_W_CM / FIG_H_CM  # x-fraction -> y-fraction, to keep panels square

fig = plt.figure(figsize=(FIG_W_CM * cm, FIG_H_CM * cm))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xticks([])
ax.set_yticks([])

fontsize_dict = {"panel": 10, "title": 8, "label": 7, "tick": 5, "legend": 5}

example_cells = [EXAMPLE_CELL, EXAMPLE_CELL2, EXAMPLE_CELL3]
n_rs_bins = 5

# --- Top row: example cells -----------------------------------------------------------
# All positions are figure fractions, measured off a rendered figure so the labels just
# clear each other (~0.006, i.e. 0.1 mm at this size). The RS/OF image spans 6 octaves of
# running speed against 12 of optic flow and imshow locks its aspect, so a matrix draws
# half as wide as it is tall: MATRIX_W is that drawn width, otherwise the axes rect would
# carry ~1.7 cm of dead space on either side of the image. MATRIX_TO_STACK clears the
# colorbar plot_RS_OF_matrix hangs off the matrix (at 1.1 x the drawn width, plus its
# rotated dF/F label) and the stack's dF/F scale bar.
X0 = 0.048  # left margin, room for the matrix y ticks and label
BAND_TOP = 0.95  # top of the row
BAND_H = 0.34  # row height, ~3.6 cm
MATRIX_W = BAND_H / ASPECT / 2  # aspect-locked drawn width
MATRIX_TO_STACK = 0.048
CELL_GAP = 0.062
STACK_W = 0.108
STACK_GAP = 0.005  # between two stacked OF tuning panels

band_bottom = BAND_TOP - BAND_H
stack_h = (BAND_H - (n_rs_bins - 1) * STACK_GAP) / n_rs_bins
block_w = MATRIX_W + MATRIX_TO_STACK + STACK_W

range_kwargs = dict(
    log_range={"log_base": 2}, rs_bins=rs_bins, of_bins=of_bins, tick_dict=tick_dict
)


def panel_ymax(ax):
    """Max y of the binned means + CI and of the fitted gaussian on one panel."""
    vals = []
    for container in ax.containers:  # errorbar: markers, caps, CI bars
        line, _, bars = container.lines
        vals.append(np.nanmax(line.get_ydata()))
        for bar in bars:
            segments = bar.get_segments()
            if len(segments):
                vals.append(max(np.nanmax(seg[:, 1]) for seg in segments))
    for line in ax.lines:  # the 300-point fit curve (not the markers or axhline)
        ydata = np.asarray(line.get_ydata(), dtype=float)
        if ydata.size > 100:
            vals.append(np.nanmax(ydata))
    return max(vals) if vals else np.nan


letters_top = ["A", "B", "C"]
for iex, (cell_uid, letter) in enumerate(zip(example_cells, letters_top)):
    block_x = X0 + iex * (block_w + CELL_GAP)
    stack_x = block_x + MATRIX_W + MATRIX_TO_STACK
    example_cell = neurons_df_treadmill[neurons_df_treadmill.roi_uid == cell_uid].iloc[
        0
    ]
    roi = example_cell.roi

    # Panel lettering
    fig.text(
        block_x - 0.038,
        0.98,
        letter,
        fontsize=fontsize_dict.get("panel", 10),
        fontweight="bold",
        ha="left",
        va="top",
    )

    # per-trial averaged rs/of/dff for this roi
    tav_df = []
    for trial, tseries in trials_df_tm.iterrows():
        ok = tseries.max_abs_rs2motor_diff_ratio_stim < max_abs_rs2motor_diff_ratio
        tav_df.append(
            dict(
                rs=tseries.RS_stim[ok].mean() * 100,
                of=np.degrees(tseries.OF_stim[ok].mean()),
                dff=tseries.dff_stim[:, roi].mean(),
            )
        )
    tav_df = pd.DataFrame(tav_df)

    # stacked OF-tuning-by-RS-bin axes, fastest RS on top
    axes_trials = [
        fig.add_axes(
            [
                stack_x,
                band_bottom + BAND_H - (i + 1) * stack_h - i * STACK_GAP,
                STACK_W,
                stack_h,
            ]
        )
        for i in range(n_rs_bins)
    ]
    for b_s, b_e, ax in zip(rs_bins[2:], rs_bins[3:], axes_trials[::-1]):
        rsof_plots.plot_rsof_slice(
            ax,
            b_s,
            b_e,
            tav_df,
            of_bins[1:],
            plot_trials=False,
            niter=10,
            color=(0.1, 0.1, 0.1),
            scatter_size=20,
            linewidth=1,
            capsize=1.5,
            markersize=3,
            fontsize_dict=fontsize_dict,
            clip_on=False,
        )
        ax.set_xticklabels([])
        ax.tick_params(axis="y", labelsize=fontsize_dict["tick"])
        ax.spines[["top", "right"]].set_visible(False)

    # One y scale per cell, rounded up to the next 0.5, shared with the matrix. The
    # panels carry no y axis: each already has a grey line at zero, and the scale is set
    # by a single bar next to the top panel, which lets the stack sit closer to its
    # matrix than the tick labels and axis label allowed.
    ytop = np.floor(max(panel_ymax(ax) for ax in axes_trials) / 0.1) * 0.1
    for iax, ax in enumerate(axes_trials):
        ax.set_ylim(-0.2, ytop * 1.2)
        ax.set_yticks([])
        ax.set_ylabel("")
        ax.spines["left"].set_visible(False)
        if iax != (len(axes_trials) - 1):
            ax.xaxis.set_visible(False)
            ax.spines["bottom"].set_visible(False)
    # dF/F scale bar: largest round value fitting in 3/4 of a panel. get_yaxis_transform
    # is x in axes fractions, y in data units, so the bar is placed just outside the left
    # edge and its length still reads in dF/F.
    bar = max(v for v in (0.1, 0.2, 0.5, 1, 2, 5) if v <= 0.75 * ytop)
    ax_bar = axes_trials[0]
    ax_bar.plot(
        [-0.03, -0.03],
        [0, bar],
        transform=ax_bar.get_yaxis_transform(),
        color="k",
        lw=1,
        clip_on=False,
    )
    ax_bar.text(
        -0.075,
        bar / 2,
        f"{bar:g} $\\Delta$F/F",
        transform=ax_bar.get_yaxis_transform(),
        rotation=90,
        ha="center",
        va="center",
        fontsize=fontsize_dict["tick"],
    )
    axes_trials[-1].set_xticks(
        [1, 10, 100, 1000],
        labels=["1", "10", "100", "1000"],
        fontsize=fontsize_dict["tick"],
    )
    axes_trials[-1].set_xlabel("Optic flow (°/s)", fontsize=fontsize_dict["label"])

    # RS/OF matrix heatmap, left of the stack and spanning the same band
    ax_matrix = fig.add_axes([block_x, band_bottom, MATRIX_W, BAND_H])
    rsof_plots.plot_RS_OF_matrix(
        trials_df=trials_df_tm,
        roi=roi,
        is_closed_loop=1,
        max_abs_rs2motor_diff_ratio=max_abs_rs2motor_diff_ratio,
        ax=ax_matrix,
        fontsize_dict=fontsize_dict,
        vmin=0,
        vmax=ytop,
        cbar_width=0.01,
        title="",
        **range_kwargs,
    )

# --- Bottom row: 2D-Gaussian fit parameters and their population distribution ----------
fig.text(
    0.008,
    0.48,
    "D",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
fig.text(
    0.585,
    0.48,
    "E",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)

if True:
    # The response is fit with a 2D Gaussian in log(RS) x log(OF); the resulting ellipse is
    # summarised by how elongated it is (eccentricity) and which way it points (orientation
    # theta). A cell tuned to the RS/OF ratio - i.e. to depth - has an elongated ellipse at
    # theta = 45 deg.
    fig.text(
        0.030,
        0.46,
        "Gaussian fit parameters",
        fontsize=fontsize_dict["title"],
        ha="left",
        va="top",
    )

    # The two sigmas are colour-coded to the axes drawn on the example fit below;
    # SEMIMAJOR_COLOR / SEMIMINOR_COLOR come from the helper that draws them.
    def draw_colored_run(fig, x, y, pieces, fontsize):
        """Draw [(text, colour), ...] left to right from (x, y) in figure coords.

        mathtext has no per-symbol colour, so the equation is a run of separate texts;
        each is measured with the renderer so the next starts where the previous ended.
        """
        renderer = fig.canvas.get_renderer()
        for string, color in pieces:
            txt = fig.text(
                x, y, string, color=color, fontsize=fontsize, ha="left", va="center"
            )
            txt.draw(renderer)
            x += txt.get_window_extent(renderer=renderer).width / fig.bbox.width
        return x

    draw_colored_run(
        fig,
        0.018,
        0.395,
        [
            ("Elongation = ", "k"),
            (r"$\sigma_{major}$", SEMIMAJOR_COLOR),
            (" / ", "k"),
            (r"$\sigma_{minor}$", SEMIMINOR_COLOR),
        ],
        fontsize_dict["label"],
    )
    draw_colored_run(
        fig,
        0.055,
        0.355,
        [("Orientation ", "k"), (r"$	heta$", "#008080")],
        fontsize_dict["label"],
    )

    # Fit schematic
    FIT_W = 0.065
    fit_h = FIT_W * ASPECT * 2
    FIT_X = 0.040
    FIT_BOTTOM = 0.065
    ax_fit = fig.add_axes([FIT_X, FIT_BOTTOM, FIT_W, fit_h])
    example_cell3 = neurons_df_treadmill[neurons_df_treadmill.roi_uid == EXAMPLE_CELL4]
    plot_g2d_fit_schematic(
        ax_fit,
        example_cell3,
        roi=example_cell3.roi.iloc[0],
        sfx=TA,
        min_sigma=MIN_SIGMA,
        fontsize_dict=fontsize_dict,
        mass_fraction=0.50,
        **range_kwargs,
    )
    ax_fit.set_title("")

# Population: orientation vs shape of every well-fit, depth-tuned neuron.
if True:
    POLAR_X, POLAR_BOTTOM, POLAR_W = 0.255, 0.055, 0.220
    POLAR_H = POLAR_W * ASPECT
    POLAR_IN_COLOR, POLAR_OUT_COLOR = "k", "darkred"
    ax_polar = fig.add_axes(
        [POLAR_X, POLAR_BOTTOM, POLAR_W, POLAR_H], projection="polar"
    )
    elongation_polar = ndf_polar[f"g2d_semimajor{TA}"].astype(float) / ndf_polar[
        f"g2d_semiminor{TA}"
    ].astype(float)
    sc_kwargs = dict(s=20, alpha=0.5, linewidths=0, clip_on=False, zorder=3)
    plot_angle_eccentricity_polar(
        ax_polar,
        ndf_polar.loc[ecc_ok, f"g2d_theta{TA}"].astype(float),
        None,
        fontsize_dict,
        radial_scale="elongation",
        axis_ratio=elongation_polar[ecc_ok],
        scale=0.5,
        rasterize_schematics="each",
        c=POLAR_IN_COLOR,
        **sc_kwargs,
    )
    ax_polar.scatter(
        np.radians(ndf_polar.loc[~ecc_ok, f"g2d_theta{TA}"].astype(float)),
        np.clip(np.log2(elongation_polar[~ecc_ok]), 0, None),
        c=POLAR_OUT_COLOR,
        **sc_kwargs,
    )

# --- Bottom row: orientation distribution with the von Mises mixture -----------------
if True:
    from v1_depth_map.figure_utils import von_mises as vm

    MAX_K = 5
    N_ANGLE_BINS = 37
    angle_edges = np.linspace(-45, 135, N_ANGLE_BINS + 1)
    angle_bin_w = np.diff(angle_edges)[0]
    angle_grid = np.linspace(-45, 135, 400)
    VONMISES_COLOR = "#000000"

    angles_deg = (
        ndf_polar.loc[ecc_ok, f"g2d_theta{TA}"].astype(float).dropna().to_numpy()
    )

    HIST_X, HIST_BOTTOM, HIST_W, HIST_H = 0.625, 0.075, 0.355, 0.355
    ax_hist = fig.add_axes([HIST_X, HIST_BOTTOM, HIST_W, HIST_H])
    ax_hist.hist(
        angles_deg, bins=angle_edges, color="lightgrey", edgecolor="k", linewidth=0.5
    )

    to_counts = len(angles_deg) * angle_bin_w

    vm_results = vm.model_selection(
        np.radians(angles_deg), max_k=MAX_K, seed=42, verbose=False
    )
    k_vm = min(vm_results, key=lambda k: vm_results[k]["bic"])
    r_vm = vm_results[k_vm]
    pi_k, mu_k, kappa_k = (
        np.asarray(r_vm["pi_k"], dtype=float),
        np.asarray(r_vm["mu_k"], dtype=float),
        np.asarray(r_vm["kappa_k"], dtype=float),
    )

    def component_counts(idx):
        """Counts curve of the components in `idx`, i.e. their weighted density."""
        return (
            vm.axial_mixture_density(
                np.radians(angle_grid), pi_k[idx], mu_k[idx], kappa_k[idx]
            )
            / np.degrees(1)
            * to_counts
        )

    ax_hist.plot(
        angle_grid,
        component_counts(slice(None)),
        color=VONMISES_COLOR,
        lw=1.2,
        zorder=3,
        label=f"von Mises mixture (k={k_vm})",
    )

    COMPONENT_COLORS = ("#E69F00", "#56B4E9", "#7F3C8D", "#009E73", "#0072B2")
    mu_deg_vm = wrap_axial_deg(np.degrees(mu_k))
    for rank, i_comp in enumerate(np.argsort(mu_deg_vm)):
        ax_hist.plot(
            angle_grid,
            component_counts([i_comp]),
            color=COMPONENT_COLORS[rank % len(COMPONENT_COLORS)],
            lw=0.8,
            ls="--",
            zorder=2,
            label=f"{mu_deg_vm[i_comp]:.0f}°: {100 * pi_k[i_comp]:.0f}%",
        )

    ax_hist.set_xlim(-45, 135)
    ax_hist.set_xticks([-45, 0, 45, 90, 135])
    ax_hist.set_xlabel("Ellipse orientation (deg)", fontsize=fontsize_dict["label"])
    ax_hist.set_ylabel("Number of cells", fontsize=fontsize_dict["label"])
    ax_hist.tick_params(axis="both", labelsize=fontsize_dict["tick"], pad=1)
    # ax_hist.spines[["top", "right"]].set_visible(False)
    ax_hist.legend(fontsize=fontsize_dict["legend"], frameon=False, loc="upper left")
    print(
        f"orientation mixture on {len(angles_deg)}/{len(ndf_polar)} neurons "
        f"(eccentricity > {ECC_CUTOFF}): von Mises k={k_vm}"
    )
    for i_comp in np.argsort(mu_deg_vm):
        print(
            f"  mu={mu_deg_vm[i_comp]:6.1f} deg  kappa={kappa_k[i_comp]:6.2f}"
            f"  {100 * pi_k[i_comp]:5.1f}% of the population"
        )
    sns.despine(ax=ax_hist, offset=0, trim=True)

style.savefig(SAVE_ROOT / "fig_depth_cells.svg", bbox_inches="tight", dpi=300)
print(f"Saved figure to {SAVE_ROOT / 'fig_depth_cells.svg'}")

In [ ]:
# Polar plot: four radial measures of the same tuning ellipses, same angles throughout.
# The first three differ only in how they compress the axis ratio a/b (a = semimajor,
# b = semiminor, so a/b >= 1 by construction); the fourth is not a shape measure at all.
#
#   flattening    1 - b/a              bounded [0, 1), roughly linear in shape
#   eccentricity  sqrt(1 - (b/a)^2)    bounded [0, 1) but saturates fast (2:1 -> 0.87)
#   log2(a/b)     unbounded, one unit per doubling of the ratio; what the paper panels use
#   sigma major   absolute size of the long axis, in ln(RS/OF) units (the g2d fit is done
#                 on natural-log RS and OF), i.e. e-folds -- 1 unit = 1.44 octaves
#
SFX = "_treadmill_trial_average_plateau"


mask = neurons_df_treadmill[f"rsof_test_rsq_closedloop_g2d{SFX}_sig"].fillna(
    False
) & neurons_df_treadmill["is_depth_neuron"].fillna(False)
ndf_polar = neurons_df_treadmill[mask].dropna(
    subset=[f"g2d_theta{SFX}", f"g2d_semimajor{SFX}", f"g2d_semiminor{SFX}"]
)
print(f"Number of neurons in polar plot: {len(ndf_polar)}/{len(neurons_df_treadmill)}")


theta = np.deg2rad(ndf_polar[f"g2d_theta{SFX}"].astype(float).to_numpy())
sigma_major = ndf_polar[f"g2d_semimajor{SFX}"].astype(float).to_numpy()
sigma_minor = ndf_polar[f"g2d_semiminor{SFX}"].astype(float).to_numpy()
# Clipped only to keep sqrt() and log2() real if a fit comes back with b marginally > a.
inv_ratio = np.clip(sigma_minor / sigma_major, 0, 1)

LOG2_RIM = 3.0  # 8:1
SIGMA_RIM = 5


def to_rim(values, rim):
    """Park non-finite and over-rim values on the rim, and say how many there were."""
    finite = np.isfinite(values)
    n_rim = int((~finite).sum() + (finite & (values > rim)).sum())
    return np.where(finite, np.minimum(values, rim), rim), n_rim


log2_ratio, n_log2_rim = to_rim(-np.log2(np.maximum(inv_ratio, 1e-300)), LOG2_RIM)
sigma_parked, n_sigma_rim = to_rim(sigma_major, SIGMA_RIM)
n_unbounded = int((~(sigma_major <= 5)).sum())
print(
    f"semimajor: {n_unbounded}/{len(sigma_major)} fits wider than the sampled grid "
    f"(> 5 ln units), largest finite value "
    f"{np.nanmax(sigma_major[np.isfinite(sigma_major)]):.3g}"
)

RADII = [
    dict(label="Flattening  $1 - b/a$", radius=1 - inv_ratio, n_rim=0, rscale="linear"),
    dict(
        label="Eccentricity  $\\sqrt{1 - (b/a)^2}$",
        radius=np.sqrt(1 - inv_ratio**2),
        n_rim=0,
        rscale="linear",
    ),
    dict(
        label=f"$\\log_2$ aspect ratio (clipped at {LOG2_RIM:.0f} = 8:1)",
        radius=log2_ratio,
        n_rim=n_log2_rim,
        rscale="linear",
    ),
    dict(
        label=f"$\\sigma_{{major}}$ (clipped at {SIGMA_RIM:.0f})",
        radius=sigma_parked,
        n_rim=n_sigma_rim,
        rscale="log",
    ),
]

fig, axes = plt.subplots(2, 2, subplot_kw=dict(projection="polar"), figsize=(5, 5))
scatter_kwargs = dict(
    s=10, alpha=0.5, linewidths=0, clip_on=False, zorder=3, color="black"
)
for ax, spec in zip(axes.flat, RADII):
    radius = spec["radius"]
    ax.scatter(theta, radius, **scatter_kwargs)
    # Orientation is axial and wrapped to [-45, 135), so only that wedge can ever hold
    # points; the full circle would leave three quarters of each panel empty.
    ax.set_thetamin(-45)
    ax.set_thetamax(135)
    ax.set_thetagrids([-45, 0, 45, 90, 135])

    ax.set_title(spec["label"], fontsize=8, pad=14)
    ax.tick_params(labelsize=6)
    rim_note = f", {spec['n_rim']} parked on the rim" if spec["n_rim"] else ""
    print(
        f"{spec['label']:48s} {np.nanmin(radius):8.2f} .. {np.nanmax(radius):10.2f}"
        f"  (median {np.nanmedian(radius):6.2f}){rim_note}"
    )
fig.suptitle(f"Radial measure comparison, {SFX}", fontsize=9)
fig.tight_layout()